## About
Accuracy assessment and area estimation of the 2024 iceplant classification map.

This notebook calculates:
- producer's accuracies and user's accuracies by class,
- overall accuracy, and
- area estimates.

Each statistic is accompanied by a 95% confidence interval. We follow the notation and calculations in Olofsson et al. (2014).

Inputs:
1. `../data/validation_points_2024.csv`: the stratified random reference points, with `map_class` (class of the point in the map) and `ref_class` (class assigned by visual interpretation of the reference imagery).
2. `../data/pixel_counts.csv`: the number of pixels per class in the map.

Class codes: 1 = iceplant, 2 = others, 3 = other vegetation.

## Reference

Pontus Olofsson, Giles M. Foody, Martin Herold, Stephen V. Stehman, Curtis E. Woodcock, Michael A. Wulder, Good practices for estimating area and assessing accuracy of land change, Remote Sensing of Environment,Volume 148, 2014, Pages 42-57, ISSN 0034-4257, https://doi.org/10.1016/j.rse.2014.02.015.  (https://www.sciencedirect.com/science/article/pii/S0034425714000704)

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels

### Load Data

In [2]:
YEAR = 2024

# class order used throughout: iceplant, others, other vegetation
CLASS_NAMES = ['iceplant', 'others', 'other vegetation']

# pixel counts of the map (sampling strata)
counts = pd.read_csv('../data/pixel_counts.csv').set_index('year')
pix_counts = counts.loc[YEAR, ['iceplant', 'others', 'other_vegetation']].tolist()
pixel_size = counts.loc[YEAR, 'pixel_size_m']

# load csv with validation points
df = pd.read_csv(f'../data/validation_points_{YEAR}.csv')

# convert class codes 1,2,3 to 0,1,2
df['class_pre'] = df['map_class'] - 1
df['class_eva'] = df['ref_class'] - 1

df.head()

,PLOTID,LON,LAT,map_class,ref_class,class_pre,class_eva
0,0,-120.470597,34.461942,1,1,0,0
1,1,-120.462015,34.448841,1,1,0,0
2,2,-120.462077,34.468564,3,3,2,2
3,3,-120.450126,34.470911,3,3,2,2
4,4,-120.474813,34.467027,2,2,1,1


In [3]:
# counts by reference class
print('Points in each reference class')
print(np.unique(df.class_eva, return_counts=True), '\n')

# counts by map class: these should match the counts given by the stratified sample design
print('Points in each map class')
print(np.unique(df.class_pre, return_counts=True))

# stratified design: 300 points each for iceplant and other vegetation, the remainder for others
assert (df.class_pre == 0).sum() == 300 and (df.class_pre == 2).sum() == 300

Points in each reference class
(array([0, 1, 2]), array([302, 113, 289])) 

Points in each map class
(array([0, 1, 2]), array([300, 104, 300]))


In [4]:
# specify number of classes in the map
n_classes = 3

# z-score for 95% confidence intervals
Z = 1.96

### Confusion Matrix

Here we create a confusion matrix $n$ such that 

$n_{i,j}$ = number of points predicted as $i$, known to be $j$, 

which is equivalent to

$n_{i,j}$ = number of points that have map class as $i$ and reference class $j$.

In [5]:
# CALCULATE CONFUSION MATRIX FOR IMPORTED DATA

# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
# using confusion_matrix directly we get a matrix C such that
# C_{i,j} = known to be i, predicted as  j 
# The notation in the paper is 
# n_{i,j} = predicted as i, known to be j 
# so we need to take the transpose

n = confusion_matrix(df.class_eva, df.class_pre, labels=range(n_classes)).T
n

array([[283,   2,  15],
       [  0, 102,   2],
       [ 19,   9, 272]])

### Notation
Throughout the following, let $p_{ij}$ be the (true) fraction of the map that has map class $i$ and reference class $j$. 

### User's Accuracy
The user's accuracy of class $i$ is the fraction of the area mapped as class $i$ that has reference class $i$, this is (Olofsson et al. eq 2):
$$U_i = \frac{p_{ii}}{p_{i\cdot}}.$$
This is equivalent to the precision of class $i$. For example, when there are two classes (positive and negative) the user's accuracy of the positive class is the same as the precision of the true class (TP/(TP + FP)).

To estimate the $U_i$'s from the points sample we have that
$$\hat{U}_i = \frac{\hat{p}_{ii}}{\hat{p}_{i\cdot}},$$
where $\hat{p}_{ij}$ are the estimations of $p_{ij}$ from the sample. 
For stratified random sampling in which the sampling strata correspond to the map classes we have that
$$\hat{p}_{ij} = W_i \frac{n_{ij}}{n_{i\cdot}},$$
where
- $W_i$ is the fraction of the map's area classified  as class i,
- $n_{ij}$ is number of points with map class $i$, known to be reference class $j$ (entires in the confusion matrix), and
- $n_{i\cdot}$ is the number of points with map class $i$ (row sums in confusion matrix).

Notice that the user's accuracy can be simplifeid to
$$\hat{U}_i = \frac{n_{ii}}{n_{i\cdot}},$$
this is the formula implemented in the code.

For user's accuracy of map class $i$, the estiamted variance is (Olfosson et al. eq. 6):
$$\hat{V}(\hat{U}_i) = \frac{\hat{U}_i (1-\hat{U}_i)}{n_{i\cdot}-1}.$$

**NOTE:** We calculate the user's accuracies first since these are needed to calculate the approximate variance of the overall accuracy. 

### Variance, Standard Error & Confidence Intervals
Recall that the square root of the estimated variance results in the standard error of the estimator. For example, in the case of the estimated overall accuracy of the map $\hat{O}$ we have that $\hat{S}(\hat{O}) = \sqrt{\hat{V}(\hat{O})}$ (see Olofsson et al. eq. 5). 

Also, the standard error is used to get confidence intervals for the estimated statistic:the 95% confidence interval is estimated as $\hat{O} \pm 1.96 \hat{S}(\hat{O}) = \hat{O} \pm 1.96\  \sqrt{\hat{V}(\hat{O})}$.

In [6]:
# -------------------------------------
# points in sample that had class i in map (predicted as i, any true class j)
# these will also be used in overal accuracy and producer's accuracies
n_idot = [sum(n[i,:]) for i in range(n_classes)]

# -------------------------------------
# estimated users' accuracy (precision for each class: TP/(TP+FP))
U_hat = [n[i,i] / n_idot[i] for i in range(n_classes)]

var_U_hat = [U_hat[i] * (1-U_hat[i])/(n_idot[i]-1) for i in range(n_classes)]

# -------------------------------------
print("user's accuracies:", [x*100 for x in U_hat])
print("user's accuracies confidence interval:", Z*np.sqrt(var_U_hat)*100)


user's accuracies: [94.33333333333334, 98.07692307692307, 90.66666666666666]
user's accuracies confidence interval: [2.62069754 2.65227898 3.2973321 ]


### Overal Accuracy

Let $O$ be the (true) accuracy of the map, and $\hat{O}$ its estimation from the sample. Then, following Olofsson et al., section 4.3, we have that 
$$\hat{O} = \sum_{i=1}^q \hat{p}_{ii},$$
where $q$ is the number of classes in the map and $\hat{p}_{ii}$ is the estimation of $p_{ij}$, the (true) fraction of the area in the map that was classified as class $i$ and has reference class $j$. 

For overall accuracy, the estimated variance is (Olofsson et al. eq 5):
$$\hat{V}(\hat{O}) = \sum_{i=1}^q \frac{W_i^2 \hat{U}_i (1-\hat{U}_i)}{n_{i\cdot}-1}.$$

In [7]:
# total number of pixels in the map
total_pix = sum(pix_counts)

# list with the fractions of area in map mapped as each class
W = [pix_counts[i]/ total_pix for i in range(n_classes)]      

# -------------------------------------
# overall accuracy
O_hat = sum([W[i]*n[i,i]/n_idot[i] for i in range(n_classes)])
print('overall accuracy:', O_hat*100)

# -------------------------------------
var_O_hat = sum([ W[i]**2 * U_hat[i] * (1-U_hat[i])/(n_idot[i]-1) for i in range(n_classes)])

# std error of estimated overall accuracy
print('overall accuracy confidence interval:', Z*np.sqrt(var_O_hat)*100, '\n')


overall accuracy: 93.48148957924866
overall accuracy confidence interval: 1.973924907873909 



### Producer's Accuracy
The producer's accuracy of class $i$ is the fraction of the (true) area with reference class $i$ that is actually mapped as class $j$, this is (Olofsson et al. eq 3):
$$P_j = \frac{p_{jj}}{p_{\cdot j}}.$$
This is equivalent to the sensitiviy of class $j$. For example, when there are two classes (positive and negative) the producer's accuracy of the positive class is the same as the sensitivy of the true class (TP/(TP + FN)).

To estimate the $P_i$'s from the points sample we have that
$$\hat{P}_j = \frac{\hat{p}_{jj}}{\hat{p}_{\cdot j}},$$
where the $\hat{p}_{ij}$ are as before.

For the producer's accuracy of class $j$ the estimated variance is given by (Olofsson et al. eq 7):

$$\hat{V}(\hat{P}_j) = 
\frac{1}{\hat{N}_{\cdot j}^2} 
\left( 
\frac{N_{j \cdot}^2 (1 - \hat{P}_j)^2 \hat{U}_j (1-\hat{U}_j)}{n_{j \cdot} -1}  
+
\hat{P}_j^2
\sum_{i\neq j}^q 
\frac{N_{i\cdot}^2}{n_{i \cdot} - 1} 
\frac{n_{ij}}{n_{i \cdot}} 
\left( 1 - \frac{n_{ij}}{n_{i \cdot}} \right)
\right),$$
where
- $N_{j \cdot}$ is the number of pixels in the map with map class $j$,
- $n_{j\cdot}$ is the number of sample points with map class $j$, and
- $\hat{N}_{\cdot j} = \sum_{i=1}^q N_{i\cdot}\frac{n_{ij}}{n_{i\cdot}}$ is the estimated number of pixels with reference class $j$.



In [8]:
p_hat_dotj = []
# estimated producer's accuracy (sensitiviy for each class TP/(TP+FN))
P_hat = []  

for j in range(n_classes):
    # list of p_hat_ij with fixed j
    p_hat_ij = [ W[i]*n[i,j]/n_idot[i] for i in range(n_classes) ]
    p_hat_dotj.append(sum(p_hat_ij))  # equation (9)


P_hat= [ (W[j]*n[j,j]/n_idot[j]) / p_hat_dotj[j] for j in range(n_classes)]
# -------------------------------------
print("producer's accuracies:", [x*100 for x in P_hat])

# -------------------------------------
# -------------------------------------
# VARIANCE
# notice N_jdot is pixel_counts[j]
N_hat_cdotj = []
for j in range(n_classes):
    summands = [ pix_counts[i] * n[i,j]/n_idot[i] for i in range(n_classes)]
    N_hat_cdotj.append(sum(summands))

# -------------------------------------
summand1 = [ (pix_counts[j]**2) * ((1-P_hat[j])**2) * U_hat[j] * (1-U_hat[j]) / (n_idot[j] - 1) 
            for j in range(n_classes)]

# -------------------------------------
summand2 = []
for j in range(n_classes):
    inner = []
    for i in range(n_classes):
        if i!=j:
            inner.append( (pix_counts[i]**2) /(n_idot[i]-1) * (n[i,j])/(n_idot[i]) * ( 1 - n[i,j]/n_idot[i]) ) 
    summand2.append((P_hat[j]**2) * sum(inner))

# -------------------------------------
var_P_hat = [1/(N_hat_cdotj[j]**2) *  (summand1[j] + summand2[j]) for j in range(n_classes)]

# -------------------------------------
# -------------------------------------
print("producer's accuracies confidence interval:", Z*np.sqrt(var_P_hat)*100, '\n')


producer's accuracies: [82.93740086787903, 94.39051137898706, 97.11513449927061]
producer's accuracies confidence interval: [6.18119792 3.22344775 1.74163954] 



### Area Estimates

For stratified random sampling when the map classes are the strata, we have that an estimator of the proportion of area of class $j$ is (Olofsson et al. eq. 9):
$$ \hat{p}_{\cdot j} = \sum_{i=1}^q W_i \frac{n_{ij}}{n_{i\cdot}}.$$

For this estimator of area proportion per class, the standard error is estimated by (Olofsson et al. eq 10):
$$S(\hat{p}_{\cdot j}) =  
\sqrt{
\sum_{i=1}^q W_i^2 \frac{ \frac{n_{ij}}{n_{i\cdot}} \left(1 -  \frac{n_{ij}}{n_{i\cdot}} \right)}{n_{i \cdot}-1}
}.$$

The estimated area of class $j$ is
$$\hat{A}_j = A \times \hat{p}_{\cdot k},$$
where $A$ is the total are of the map. 
The standard error for the area is given by (Olofsson et al. eq 11):
$$ S(\hat{A}_j) = A \times S(\hat{p}_{\cdot j}).$$

In [9]:
# PERCENTAGE OF AREA ESTIMATION
# we had calculated the area estimators before, they are used in producer's accuracy
print("percentage of area per class: \n", [x*100 for x in p_hat_dotj])

# -------------------------------------
# STD ERROR
SE_p_hat_dotj = []
for j in range(n_classes):
    summands = [ (W[i]**2) * (n[i,j]/n_idot[i]) * (1 -  (n[i,j]/n_idot[i]))/ (n_idot[i]-1) 
                for i in range(n_classes)]
    SE_p_hat_dotj.append(np.sqrt(sum(summands)))
    
print("confidence interval for percentage area per class:\n", [x*Z*100 for x in SE_p_hat_dotj])


percentage of area per class: 
 [19.760819961468044, 30.536645465284607, 49.70253457324736]
confidence interval for percentage area per class:
 [1.538670582441573, 1.3011264210767381, 1.9680658466435283]


In [10]:
map_area = total_pix * pixel_size**2  # m^2

approx_area_per_class = [map_area * p_hat_dotj[i] for i in range(n_classes)]
print("approx area per class (ha): \n", [x / 1e4 for x in approx_area_per_class])

SE_area_per_class = [map_area * SE_p_hat_dotj[i] for i in range(n_classes)]
print("confidence interval for area per class (ha):\n", [x * Z / 1e4 for x in SE_area_per_class])

approx area per class (ha): 
 [211.33576379999997, 326.5798335230769, 531.5529986769232]
confidence interval for area per class (ha):
 [16.455598675102912, 13.915138467674437, 21.04784617845405]


### Summary

In [11]:
summary = pd.DataFrame({
    'class': CLASS_NAMES,
    'UA (%)': [100 * x for x in U_hat],
    'UA CI (%)': Z * np.sqrt(var_U_hat) * 100,
    'PA (%)': [100 * x for x in P_hat],
    'PA CI (%)': Z * np.sqrt(var_P_hat) * 100,
    'area (%)': [100 * x for x in p_hat_dotj],
    'area CI (%)': [100 * Z * x for x in SE_p_hat_dotj],
    'area (ha)': [x / 1e4 for x in approx_area_per_class],
    'area CI (ha)': [Z * x / 1e4 for x in SE_area_per_class],
})
summary.insert(0, 'year', YEAR)
summary['OA (%)'] = O_hat * 100
summary['OA CI (%)'] = Z * np.sqrt(var_O_hat) * 100

os.makedirs('../results', exist_ok=True)
summary.to_csv(f'../results/accuracy_{YEAR}.csv', index=False)
summary.round(2)

,year,class,UA (%),UA CI (%),PA (%),PA CI (%),area (%),area CI (%),area (ha),area CI (ha),OA (%),OA CI (%)
0,2024,iceplant,94.33,2.62,82.94,6.18,19.76,1.54,211.34,16.46,93.48,1.97
1,2024,others,98.08,2.65,94.39,3.22,30.54,1.30,326.58,13.92,93.48,1.97
2,2024,other vegetation,90.67,3.30,97.12,1.74,49.70,1.97,531.55,21.05,93.48,1.97
